# Lesson 13: SLMOps & Knowledge Distillation — Transferring Gemini Teacher Reasoning to Edge Student SLMs

In this companion notebook, you will:
1. **Stage 1 (Teacher Synthetic Data Generation)**: Use **Google AI Studio Free Tier** (`gemini-2.5-flash` via `google-genai`) with an automatic offline fallback to generate structured Chain-of-Thought training examples.
2. **Stage 2 (Student Distillation Loss)**: Compute temperature-scaled Kullback-Leibler (KL) divergence + hard-label Cross-Entropy loss in PyTorch.
3. Fine-tune a compact student checkpoint (`HuggingFaceTB/SmolLM2-135M-Instruct`) on the distilled dataset.

In [ ]:
!pip install -q -U google-genai transformers torch

## 1. Stage 1: Generate Synthetic Reasoning Data with Gemini Teacher (with Offline Fallback)
Add `GEMINI_API_KEY` to Colab **Secrets** from [aistudio.google.com/apikey](https://aistudio.google.com/apikey).

In [ ]:
import os, json

GEMINI_API_KEY = None
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

prompts = [
    "Classify sensor alert: 'Motor temp 98C, vibration 4.2mm/s' into NORMAL, WARN, or CRITICAL with one-sentence rationale.",
    "Classify sensor alert: 'Motor temp 62C, vibration 0.8mm/s' into NORMAL, WARN, or CRITICAL with one-sentence rationale."
]

synthetic_dataset = []
if GEMINI_API_KEY:
    from google import genai
    client = genai.Client(api_key=GEMINI_API_KEY)
    for p in prompts:
        resp = client.models.generate_content(model="gemini-2.5-flash", contents=p)
        synthetic_dataset.append({"prompt": p, "teacher_completion": resp.text.strip()})
else:
    print("GEMINI_API_KEY not found — using deterministic pre-distilled synthetic traces.")
    synthetic_dataset = [
        {"prompt": prompts[0], "teacher_completion": "CRITICAL: Temperature exceeds 90C threshold and vibration is elevated."},
        {"prompt": prompts[1], "teacher_completion": "NORMAL: Both temperature and vibration remain within nominal operating limits."}
    ]

print(json.dumps(synthetic_dataset, indent=2))

## 2. Stage 2: Temperature-Scaled KL-Divergence Distillation Loss
Compute the combined distillation objective $\mathcal{L}_{\text{KD}} = (1-\alpha)\mathcal{L}_{\text{CE}} + \alpha T^2 D_{\text{KL}}(\sigma(z_T/T) \| \sigma(z_S/T))$ in PyTorch.

In [ ]:
import torch
import torch.nn.functional as F

teacher_logits = torch.tensor([[4.2, 1.1, -0.5]])
student_logits = torch.tensor([[2.1, 1.4,  0.2]], requires_grad=True)
hard_label = torch.tensor([0])

T = 2.0
alpha = 0.7
ce_loss = F.cross_entropy(student_logits, hard_label)
kl_loss = F.kl_div(
    F.log_softmax(student_logits / T, dim=-1),
    F.softmax(teacher_logits / T, dim=-1),
    reduction="batchmean"
) * (T ** 2)
total_loss = (1.0 - alpha) * ce_loss + alpha * kl_loss
total_loss.backward()
print(f"CE Loss: {ce_loss.item():.4f} | KL Loss: {kl_loss.item():.4f} | Total KD Loss: {total_loss.item():.4f}")